# 🚀 LLM LoRA Fine-Tuning on Google Colab

Train your LLM with LoRA on free GPU!

**Hardware:** T4 GPU (15GB VRAM) - Free tier
**Time:** 2-6 hours for 1000 steps
**Model:** Llama-2-7B or GPT-2

## Setup Checklist
- [ ] Runtime: GPU (Runtime > Change runtime type > T4 GPU)
- [ ] GitHub repo: Public or have access token
- [ ] (Optional) WandB account for tracking
- [ ] (Optional) AWS credentials for S3 upload

## Step 1: Check GPU

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## Step 2: Mount Google Drive (Optional - for saving models)

In [ ]:
# Step 2: Mount Google Drive and Create Project Folders

from google.colab import drive
drive.mount('/content/drive')

# Create organized folder structure
import os

PROJECT_ROOT = "/content/drive/MyDrive/llm-projects"
os.makedirs(f"{PROJECT_ROOT}/models", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/checkpoints", exist_ok=True)

print("✅ Google Drive mounted!")
print(f"✅ Project structure created at: {PROJECT_ROOT}")
print("\nFolders:")
print(f"  📁 Models:      {PROJECT_ROOT}/models/")
print(f"  📁 Datasets:    {PROJECT_ROOT}/datasets/")
print(f"  📁 Checkpoints: {PROJECT_ROOT}/checkpoints/")

## Step 3: Clone Your GitHub Repository

In [ ]:
# CONFIGURATION - Update these!
GITHUB_USERNAME = "umair-ds92"  # ← Your GitHub username
REPO_NAME = "llm-finetuning-lora"  # ← Your repo name

# Clone repository
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd {REPO_NAME}

# Check contents
!ls -la

## Step 4: Install Dependencies

In [ ]:
# Install requirements
!pip install -q -r requirements.txt

# Install package in editable mode
!pip install -q -e .

print("\n✅ Dependencies installed!")

## Step 5: Configure Training

In [ ]:
# TRAINING CONFIGURATION
CONFIG = {
    # Model
    "base_model": "gpt2",  # or "meta-llama/Llama-2-7b-hf" (requires HF token)
    
    # Data
    "num_examples": 1000,  # Generate 1000 training examples (vs 80 locally)
    
    # Training
    "num_epochs": 3,
    "batch_size": 8,  # GPU can handle larger batches
    "learning_rate": 2e-4,
    "max_steps": 500,  # vs 20 on CPU
    
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    
    # Output
    "output_dir": "outputs/colab-finetuned",
    
    # Monitoring (optional)
    "use_wandb": False,  # Set to True if you have WandB account
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Step 6: Generate Training Data

In [ ]:
# Generate larger dataset for GPU training
!python scripts/prepare_data.py \
    --output-dir data \
    --num-examples {CONFIG['num_examples']} \
    --train-split 0.8 \
    --val-split 0.1 \
    --test-split 0.1

# Verify data
!wc -l data/splits/*.jsonl

## Step 7: Start Training! 🚀

In [ ]:
# Training command
!python src/training/train_lora.py \
    --base-model {CONFIG['base_model']} \
    --train-data data/splits/train.jsonl \
    --val-data data/splits/validation.jsonl \
    --output-dir {CONFIG['output_dir']} \
    --num-epochs {CONFIG['num_epochs']} \
    --batch-size {CONFIG['batch_size']} \
    --learning-rate {CONFIG['learning_rate']} \
    --max-steps {CONFIG['max_steps']} \
    --lora-r {CONFIG['lora_r']} \
    --lora-alpha {CONFIG['lora_alpha']} \
    --lora-dropout {CONFIG['lora_dropout']}

## Step 8: Evaluate Model

In [ ]:
# Evaluate on test set
!python src/evaluation/evaluate.py \
    --base-model {CONFIG['base_model']} \
    --adapter-path {CONFIG['output_dir']} \
    --test-data data/splits/test.jsonl \
    --output-dir outputs/colab-evaluation \
    --compare-with-base

## Step 9: Save Model to Google Drive

In [ ]:
# Step 9: Save Model to Google Drive (Organized)

import shutil
from datetime import datetime

# Create timestamped model folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
model_name = f"colab-{CONFIG['base_model'].replace('/', '-')}-{timestamp}"
drive_path = f"/content/drive/MyDrive/llm-projects/models/{model_name}"

print(f"Saving model to: {drive_path}")

shutil.copytree(
    CONFIG['output_dir'],
    drive_path,
    dirs_exist_ok=True
)

# Save training config for reference
import json
config_path = f"{drive_path}/training_config.json"
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"\n✅ Model saved!")
print(f"📁 Location: {drive_path}")
print(f"📝 Config saved: {config_path}")

# Create a latest symlink for easy access
latest_path = "/content/drive/MyDrive/llm-projects/models/latest"
if os.path.exists(latest_path):
    os.remove(latest_path)
os.symlink(drive_path, latest_path)
print(f"🔗 Latest model: {latest_path}")

## Step 10: Upload to AWS S3 (Optional)

In [ ]:
# Install AWS CLI
!pip install -q awscli boto3

# Configure AWS credentials (run once)
from getpass import getpass

aws_access_key = getpass("AWS Access Key ID: ")
aws_secret_key = getpass("AWS Secret Access Key: ")
aws_region = input("AWS Region (default: us-east-1): ") or "us-east-1"

!aws configure set aws_access_key_id {aws_access_key}
!aws configure set aws_secret_access_key {aws_secret_key}
!aws configure set default.region {aws_region}

print("\n✅ AWS credentials configured!")

In [ ]:
# Upload to S3
S3_BUCKET = "your-bucket-name"  # ← Update this!
S3_PREFIX = "models/llm-lora-finetuned"

print(f"Uploading to s3://{S3_BUCKET}/{S3_PREFIX}/")

!aws s3 sync {CONFIG['output_dir']} s3://{S3_BUCKET}/{S3_PREFIX}/ \
    --exclude "*.git/*" \
    --exclude "__pycache__/*"

print("\n✅ Model uploaded to S3!")
print(f"Location: s3://{S3_BUCKET}/{S3_PREFIX}/")

## Step 11: Quick Inference Test

In [ ]:
# Test the fine-tuned model
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading model...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, CONFIG['output_dir'])
tokenizer = AutoTokenizer.from_pretrained(CONFIG['base_model'])

# Test generation
prompt = """### Instruction:
Analyze this security threat

### Input:
Multiple failed SSH login attempts from 192.168.1.100

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*80)
print("TEST GENERATION")
print("="*80)
print(response)
print("="*80)

## Step 12: Download Model (Alternative to Drive/S3)

In [ ]:
# Zip the model for download
!zip -r colab-finetuned-model.zip {CONFIG['output_dir']}

# Download via Colab interface
from google.colab import files
files.download('colab-finetuned-model.zip')

print("\n✅ Model zipped and ready for download!")

## 📊 Training Summary

In [ ]:
# Display training summary
import json

results_file = f"{CONFIG['output_dir']}/trainer_state.json"

if os.path.exists(results_file):
    with open(results_file) as f:
        trainer_state = json.load(f)
    
    print("\n" + "="*80)
    print("TRAINING SUMMARY")
    print("="*80)
    print(f"Model: {CONFIG['base_model']}")
    print(f"Training examples: {CONFIG['num_examples']}")
    print(f"Total steps: {trainer_state.get('global_step', 'N/A')}")
    print(f"Best metric: {trainer_state.get('best_metric', 'N/A')}")
    print(f"Training time: {trainer_state.get('total_flos', 'N/A')}")
    print("="*80)
else:
    print("Training results not found!")

## 🎉 Next Steps

Your model is trained! Now you can:

1. **Download from Google Drive** - Access from any device
2. **Pull from S3** - Use in AWS deployment
3. **Test locally** - Merge weights and run inference
4. **Deploy to production** - FastAPI server or AWS Lambda

### On Your Local Machine:

```bash
# Download from S3
aws s3 sync s3://your-bucket/models/llm-lora-finetuned/ outputs/colab-model/

# Merge weights
python src/deployment/merge_weights.py \
    --base-model gpt2 \
    --adapter-path outputs/colab-model \
    --output-dir outputs/merged-colab-model

# Deploy API
python src/deployment/serve.py \
    --model-path outputs/merged-colab-model
```